# PE6201 Final Project — Final Evaluation v3 (No Colab Secrets)

This notebook supports the final evaluation of the **Badminton AI Coach**.

It:
1. reproduces the fixed classifier split and local baseline;
2. loads the 20-note badminton knowledge base and retrieval layer;
3. can rerun the 10 fixed end-to-end advice cases through OpenRouter;
4. records automatic diagnostics without overwriting the preserved final evidence;
5. includes the **human marks from the preserved final measured run**.

**Security:** the OpenRouter key is entered through hidden input and is never written into the notebook.

**Important:** the human marks below correspond to the preserved final outputs in `results/advice_eval_final.csv`. If you intentionally rerun the LLM and obtain different outputs, those new outputs must be judged again rather than reusing the preserved marks.


In [ ]:
%pip -q install openai scikit-learn sentence-transformers pandas fastapi uvicorn

## 0. Load the submitted GitHub repository

In [ ]:
import os, subprocess, sys, json, csv, time
from pathlib import Path

REPO_DIR = Path("/content/Badminton-ai-coach-PE6201")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/Keira11YIZHEN/Badminton-ai-coach-PE6201.git", str(REPO_DIR)],
        check=True
    )

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
print("Top-level files:", sorted(os.listdir("."))[:20])


## 1. Fixed data and classifier baseline

In [ ]:
from collections import Counter
from sklearn.metrics import accuracy_score
from APP.classifier import load_data, train_classifier

rows = load_data()
model, (X_train, X_test, y_train, y_test) = train_classifier(rows)

majority = Counter(y_test).most_common(1)[0][0]
baseline = [majority] * len(y_test)

t0 = time.perf_counter()
pred = model.predict(X_test)
latency = (time.perf_counter() - t0) / len(X_test) * 1000

print(f"Total: {len(rows)} | train: {len(X_train)} | test: {len(X_test)}")
print(f"Test support: {dict(Counter(y_test))}")
print(f"Majority baseline: {accuracy_score(y_test, baseline):.1%}")
print(f"TF-IDF accuracy: {accuracy_score(y_test, pred):.1%} ({sum(a==b for a,b in zip(pred,y_test))}/{len(y_test)})")
print(f"TF-IDF latency this run: {latency:.3f} ms/item")


## 2. Knowledge base and retrieval

In [ ]:
from APP.retrieval import load_notes, Retriever

notes = load_notes()
print("Knowledge-base notes:", len(notes))
print("Category counts:", dict(Counter(n["category"] for n in notes)))

retriever = Retriever()
print("Retrieval backend:", retriever.using)

sample_hits = retriever.retrieve(
    "My net shot keeps touching the tape because I push too hard.",
    category="technique",
    top_k=3,
)
sample_hits


## 3. Grounded advisor setup

The generator uses GPT-4o-mini through OpenRouter **only after local classification and local retrieval**.  
It is instructed to use only the retrieved notes and to abstain when the notes do not support a useful answer.


In [ ]:
import os
from getpass import getpass

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip()

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = getpass("Paste your OpenRouter API key (input hidden): " ).strip()
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

if not OPENROUTER_API_KEY:
    raise ValueError("No OpenRouter API key was entered.")

print("API key loaded for this runtime only; it is not displayed or saved.")


In [ ]:
from APP.pipeline import BadmintonCoach

coach = BadmintonCoach(top_k=3, api_key=OPENROUTER_API_KEY)
print("BadmintonCoach pipeline is ready.")


## 4. Optional rerun of the ten fixed end-to-end advice cases

This cell reruns the same fixed 10-case evaluation **without overwriting the preserved final evidence**.

- `category_correct_0_1`: routing correctness
- `evidence_hit_0_1`: whether required evidence appears in Top-3
- `abstained_0_1`: whether the model returns ‘The notes do not say’
- token counts: OpenRouter usage

Because LLM outputs can vary across runs, rerun outputs are saved separately as `advice_eval_outputs_rerun.csv`.


In [ ]:
import pandas as pd

cases = pd.read_csv("evals/advice_eval.csv")
rerun_rows = []
token_in = 0
token_out = 0

for i, row in cases.iterrows():
    out = coach.run(row["input"])
    retrieved_ids = [h["id"] for h in out["retrieved"]]
    advice = out["advice"]
    usage = out.get("usage") or {}

    pt = usage.get("prompt_tokens") or 0
    ct = usage.get("completion_tokens") or 0
    token_in += pt
    token_out += ct

    drill = str(advice.get("drill", "")) if isinstance(advice, dict) else ""

    rerun_rows.append({
        "case_id": row["case_id"],
        "input": row["input"],
        "expected_category": row["expected_category"],
        "required_evidence": row["required_evidence"],
        "model_category": out["category"],
        "category_correct_0_1": int(out["category"] == row["expected_category"]),
        "classifier_confidence": round(out["classifier_confidence"], 4),
        "retrieved_note_ids": ";".join(retrieved_ids),
        "evidence_hit_0_1": int(row["required_evidence"] in retrieved_ids),
        "model_advice": json.dumps(advice, ensure_ascii=False),
        "prompt_tokens": pt,
        "completion_tokens": ct,
        "abstained_0_1": int("notes do not say" in drill.lower()),
    })

rerun_df = pd.DataFrame(rerun_rows)
rerun_df.to_csv("advice_eval_outputs_rerun.csv", index=False)

print("RERUN AUTOMATIC DIAGNOSTICS")
print("Route correctness:", int(rerun_df["category_correct_0_1"].sum()), "/10")
print("Evidence recall@3:", int(rerun_df["evidence_hit_0_1"].sum()), "/10")
print("Abstentions:", int(rerun_df["abstained_0_1"].sum()), "/10")
print("Tokens:", token_in, "input /", token_out, "output")


## 5. HAND MARK — preserved final measured run

These marks are the student's human judgement for the **preserved final measured outputs** stored in `results/advice_eval_final.csv`.

Scoring:
- **Specific = 1** only if the drill gives a concrete action.
- **Supported = 1** only if the advice is supported by the retrieved notes.
- **No invention = 1** only if it adds no unsupported timing, repetitions, diagnosis, safety claim, or factual detail.

These are human marks, not an LLM judge.


In [ ]:
MANUAL_MARKS = {
    'A01': (1, 1, 1, 'Specific and grounded in doc0.'),
    'A02': (1, 1, 1, 'Specific and grounded, although routing was incorrect.'),
    'A03': (1, 1, 1, 'Specific clear advice supported by retrieved notes.'),
    'A04': (0, 0, 1, 'Wrong routing prevented useful evidence retrieval; model abstained.'),
    'A05': (0, 0, 1, 'Relevant evidence was retrieved, but the model abstained unnecessarily.'),
    'A06': (0, 0, 1, 'Issue was supported, but no actionable drill was provided.'),
    'A07': (1, 1, 1, 'Specific net-shot correction supported by doc7.'),
    'A08': (1, 1, 1, 'Specific footwork advice supported by retrieved notes.'),
    'A09': (0, 0, 1, 'Wrong routing caused irrelevant retrieval and abstention.'),
    'A10': (1, 1, 1, 'Specific shadow-footwork drill supported by doc19.'),
}

MANUAL_MARKS


## 6. Verify preserved final summary

In [ ]:
with open("results/final_eval_summary.json", encoding="utf-8") as f:
    preserved_summary = json.load(f)

specific = sum(v[0] for v in MANUAL_MARKS.values())
supported = sum(v[1] for v in MANUAL_MARKS.values())
no_invention = sum(v[2] for v in MANUAL_MARKS.values())

print("Human marks from MANUAL_MARKS")
print(f"Specificity: {specific}/10 = {specific/10:.0%}")
print(f"Supported by notes: {supported}/10 = {supported/10:.0%}")
print(f"No unsupported invention: {no_invention}/10 = {no_invention/10:.0%}")

print("\nPreserved final automatic + human summary")
print(json.dumps(preserved_summary, indent=2))

assert specific == preserved_summary["specific_passes"]
assert supported == preserved_summary["supported_passes"]
assert no_invention == preserved_summary["no_invention_passes"]

print("\nVerification passed: MANUAL_MARKS match the preserved final summary.")
